### LLaMA Supervised Fine-Tuning

This document will perform the inference on evaluation dataset of freedom intelligence using the base model

In [1]:
import os

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
import pandas as pd
import json
import torch
import pickle
from unsloth import FastLanguageModel
from datasets import Dataset
from tqdm  import tqdm
import evaluate

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/data/mn27889/miniconda3/envs/mental-health-agents/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!


#### Reading the Question and Answer Pairs from Test Dataset Phase 2

In [4]:
ques_list = []
ans_list = []
llama_resp_list = []
gpt_resp_list = []

with open('phase2_data_freedom_intelligence/test_freedom_intelligence.jsonl', 'rb') as file:
    for line in file:
        json_object = json.loads(line)
        ques_list.append(json_object['question'])
        ans_list.append(json_object['answer'])

In [5]:
test_dataset = pd.DataFrame({'question': ques_list,
                          'answer': ans_list})
test_dataset

,question,answer
0,A 59-year-old man has a 5-month history of bre...,Subpleural cystic enlargement
1,A patient presents with photopsia and floaters...,Rhegmetogenous retinal detachment
2,Based on the clinical presentation and the pat...,Drug-induced pulmonary disease
3,A 51-year-old woman presents with weakness tha...,Type II hypersensitivity reaction
4,What clinical finding is most likely to be pre...,Canon A waves
...,...,...
4060,Which anticoagulant does not require routine c...,Dabigatran etexilate
4061,A child presents with perianal itching that di...,Enterobius vermicularis
4062,What is the appropriate investigation to diagn...,Transcranial ultrasound
4063,A 12-month-old boy presents with a history of ...,NADPH oxidase complex


### Inference

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = 1024, # [NEW!] Max sequence length for the model
    load_in_4bit = False, # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    dtype=None, #None for auto-detection. Can be torch.bfloat16 or torch.float16 (will be automatically detected)
    device_map="auto"
)

Implementing sample-by-sample inference. (Batch Inference doesn't work well for fine-tuned model adapters as responses like `P P P P` are being produced)

In [6]:
def get_llama_response(question_input: str):
    
    llama_input = [{"role": "system", "content": "You are a medical knowledge assistant trained to provide information and guidance on various health-related topics."},
                    {"role": "user", "content": question_input}]

    prompt = tokenizer.apply_chat_template(llama_input, tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer(prompt, padding=True, truncation=True, return_tensors="pt").to(model.device)
    temp_resp = tokenizer.decode(inputs['input_ids'][0], skip_special_tokens=True)
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens=1024,
        num_return_sequences=1
    )

    resp = tokenizer.decode(outputs[0], skip_special_tokens=True)
    resp = resp[len(temp_resp):] #getting only the response part (i.e., assistant)
    
    return resp

In [7]:
# # Implementing the Unsloth Fast Inference
# FastLanguageModel.for_inference(model)

# llama_responses_base = []
# for index, row in tqdm(test_dataset.iterrows(), total=len(test_dataset)):
#     question_input = row['question']
#     llama_resp = get_llama_response(question_input)
#     llama_responses_base.append(llama_resp)

# with open('phase2_freedom_intelligence/llama_responses_base.pkl', 'wb') as file:
#     pickle.dump(llama_responses_base, file)

In [8]:
with open('phase2_freedom_intelligence/llama_responses_base.pkl', 'rb') as file:
    llama_responses_base = pickle.load(file)

### Saving the LLaMA Responses into the complete dataframe

In [9]:
test_dataset['llama_responses_base'] = llama_responses_base
test_dataset

,question,answer,llama_responses_base
0,A 59-year-old man has a 5-month history of bre...,Subpleural cystic enlargement,Based on the patient's history and physical ex...
1,A patient presents with photopsia and floaters...,Rhegmetogenous retinal detachment,Given the patient's symptoms of photopsia (fla...
2,Based on the clinical presentation and the pat...,Drug-induced pulmonary disease,Based on the clinical presentation of diffuse ...
3,A 51-year-old woman presents with weakness tha...,Type II hypersensitivity reaction,"Based on the symptoms described, the most like..."
4,What clinical finding is most likely to be pre...,Canon A waves,Based on the clinical presentation and history...
...,...,...,...
4060,Which anticoagulant does not require routine c...,Dabigatran etexilate,Warfarin does not require routine coagulation ...
4061,A child presents with perianal itching that di...,Enterobius vermicularis,"Based on the symptoms and findings presented, ..."
4062,What is the appropriate investigation to diagn...,Transcranial ultrasound,In a 2-day-old premature neonate who develops ...
4063,A 12-month-old boy presents with a history of ...,NADPH oxidase complex,"Based on the provided information, the most li..."


### Calculating the BLEU Results for Phase 3

LLaMA Response Groundtruth Fine-Tuned

In [10]:
bleu_eval = evaluate.load("bleu")
bleu_results = bleu_eval.compute(predictions=test_dataset['llama_responses_base'].to_list(), references=test_dataset['answer'].to_list())
bleu_results

{'bleu': 0.0020714821519528513,
 'precisions': [0.009866665317552065,
  0.002957116722202658,
  0.0011424744924039504,
  0.000552381857724418],
 'brevity_penalty': 1.0,
 'length_ratio': 48.931922267607376,
 'translation_length': 790642,
 'reference_length': 16158}

### Calculating the ROUGE Results for Phase 3

LLaMA Response Fine-Tuned

In [11]:
rouge_eval = evaluate.load("rouge")
rouge_results = rouge_eval.compute(predictions=test_dataset['llama_responses_base'].to_list(), references=test_dataset['answer'].to_list())
rouge_results

{'rouge1': np.float64(0.031520593776073504),
 'rouge2': np.float64(0.009662466549234128),
 'rougeL': np.float64(0.028934365916532506),
 'rougeLsum': np.float64(0.029536284517701454)}